# 1. Исследование данных (EDA)\n\nАнализ ИК-датасета для детекции огня с БПЛА:\n- Визуализация термальных изображений\n- Распределение классов и размеров bbox\n- Анализ интенсивности пикселей\n- Примеры аннотаций

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
CLASS_NAMES = {0: 'fire', 1: 'smoke', 2: 'false_source'}
CLASS_COLORS_RGB = {0: 'red', 1: 'gold', 2: 'dodgerblue'}

## 1.1 Статистика датасета

In [ ]:
def get_dataset_stats(split: str):
    """Собирает статистику по split."""
    img_dir = PROCESSED_DIR / split / 'images'
    lbl_dir = PROCESSED_DIR / split / 'labels'
    
    images = list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg'))
    
    class_bbox_counts = Counter()
    bbox_sizes = []  # (class_id, w, h)
    images_per_class = Counter()
    total_bboxes = 0
    negative_count = 0
    
    for img_file in images:
        lbl_file = lbl_dir / (img_file.stem + '.txt')
        if not lbl_file.exists():
            negative_count += 1
            continue
            
        content = lbl_file.read_text().strip()
        if not content:
            negative_count += 1
            continue
        
        img_classes = set()
        for line in content.split('\n'):
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                w, h = float(parts[3]), float(parts[4])
                class_bbox_counts[cls_id] += 1
                bbox_sizes.append((cls_id, w, h))
                img_classes.add(cls_id)
                total_bboxes += 1
        
        for c in img_classes:
            images_per_class[c] += 1
    
    return {
        'n_images': len(images),
        'n_negative': negative_count,
        'class_bbox_counts': class_bbox_counts,
        'images_per_class': images_per_class,
        'bbox_sizes': bbox_sizes,
        'total_bboxes': total_bboxes,
    }

# Сбор статистики
stats = {}
for split in ['train', 'val', 'test']:
    stats[split] = get_dataset_stats(split)
    s = stats[split]
    print(f"\n{'='*40}")
    print(f"{split.upper()}: {s['n_images']} изображений, {s['total_bboxes']} bbox")
    print(f"  Negative (без объектов): {s['n_negative']}")
    for cls_id, name in CLASS_NAMES.items():
        n_bbox = s['class_bbox_counts'].get(cls_id, 0)
        n_img = s['images_per_class'].get(cls_id, 0)
        print(f"  {name}: {n_bbox} bbox в {n_img} изображениях")

## 1.2 Распределение классов

In [ ]:
# Диаграмма распределения классов по splits
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, split in enumerate(['train', 'val', 'test']):
    s = stats[split]
    names = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES.keys())]
    counts = [s['class_bbox_counts'].get(i, 0) for i in sorted(CLASS_NAMES.keys())]
    colors = [CLASS_COLORS_RGB[i] for i in sorted(CLASS_NAMES.keys())]
    
    axes[idx].bar(names, counts, color=colors, edgecolor='black')
    axes[idx].set_title(f'{split.upper()} ({s["n_images"]} img)')
    axes[idx].set_ylabel('Количество bbox')
    
    for j, c in enumerate(counts):
        axes[idx].text(j, c + max(counts)*0.02, str(c), ha='center', fontweight='bold')

plt.suptitle('Распределение классов по splits', fontsize=14)
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'runs' / 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 1.3 Распределение размеров bbox

In [ ]:
# Размеры bbox (нормализованные → пиксели при 640x512)
IMG_W, IMG_H = 640, 512

all_sizes = stats['train']['bbox_sizes']
if all_sizes:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    for cls_id, name in CLASS_NAMES.items():
        cls_sizes = [(w * IMG_W, h * IMG_H) for c, w, h in all_sizes if c == cls_id]
        if not cls_sizes:
            continue
        widths, heights = zip(*cls_sizes)
        areas = [w * h for w, h in cls_sizes]
        
        axes[0].hist(widths, bins=30, alpha=0.6, label=name, color=CLASS_COLORS_RGB[cls_id])
        axes[1].hist(heights, bins=30, alpha=0.6, label=name, color=CLASS_COLORS_RGB[cls_id])
        axes[2].hist(areas, bins=30, alpha=0.6, label=name, color=CLASS_COLORS_RGB[cls_id])
    
    axes[0].set_title('Ширина bbox (px)')
    axes[1].set_title('Высота bbox (px)')
    axes[2].set_title('Площадь bbox (px²)')
    
    # COCO size thresholds
    axes[2].axvline(x=32**2, color='gray', linestyle='--', label='small/medium (32²)')
    axes[2].axvline(x=96**2, color='gray', linestyle=':', label='medium/large (96²)')
    
    for ax in axes:
        ax.legend()
        ax.set_ylabel('Количество')
    
    plt.suptitle('Распределение размеров bbox (train)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Нет данных — сначала подготовьте датасет")

## 1.4 Примеры изображений с аннотациями

In [ ]:
import random

def show_samples(split='train', n=8, seed=42):
    """Показать случайные изображения с bbox."""
    random.seed(seed)
    img_dir = PROCESSED_DIR / split / 'images'
    lbl_dir = PROCESSED_DIR / split / 'labels'
    
    images = list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg'))
    if not images:
        print(f"Нет изображений в {img_dir}")
        return
    
    selected = random.sample(images, min(n, len(images)))
    cols = 4
    rows = (len(selected) + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
    axes = axes.flatten() if rows > 1 else [axes] if rows == 1 and cols == 1 else axes.flatten()
    
    color_map = {0: (255, 0, 0), 1: (255, 215, 0), 2: (30, 144, 255)}
    
    for idx, img_file in enumerate(selected):
        img = cv2.imread(str(img_file))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        lbl_file = lbl_dir / (img_file.stem + '.txt')
        if lbl_file.exists():
            for line in lbl_file.read_text().strip().split('\n'):
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = [float(x) for x in parts[1:5]]
                    x1 = int((xc - bw/2) * w)
                    y1 = int((yc - bh/2) * h)
                    x2 = int((xc + bw/2) * w)
                    y2 = int((yc + bh/2) * h)
                    color = color_map.get(cls_id, (255, 255, 255))
                    cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img_rgb, CLASS_NAMES.get(cls_id, '?'), (x1, y1-5),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        
        axes[idx].imshow(img_rgb)
        axes[idx].set_title(img_file.name, fontsize=9)
        axes[idx].axis('off')
    
    for idx in range(len(selected), len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f'Примеры из {split} split', fontsize=14)
    plt.tight_layout()
    plt.show()

show_samples('train', n=8)